# The goal is to collect all the data form the different years and compare the best algos from each year to determine the best of the best : BBOB- biobj test Suite

We first build a table where we will agregate all the data from all of the different years.

In [5]:
import pandas as pd
from pathlib import Path

# === 1. Load all yearly CSVs from the "results" folder ===
folder = Path("results")
all_files = sorted(folder.glob("bbob-biobj_*.csv"))

dfs = []
for f in all_files:
    year = int(f.stem.split("_")[-1])
    df = pd.read_csv(f)
    df["year"] = year
    dfs.append(df)

# Merge everything into a single DataFrame
df_all = pd.concat(dfs, ignore_index=True)

# === 2. Find, for each (dim, func, target), the entry with smallest ERT ===
idx = df_all.groupby(["dimension", "function_id", "target"])["best_ERT"].idxmin()
df_best_overall = df_all.loc[idx, ["dimension", "function_id", "year", "best_algorithm", "target", "best_ERT"]]

# Sort nicely
df_best_overall = df_best_overall.sort_values(["dimension", "function_id", "target"]).reset_index(drop=True)

# === 3. Display result ===
print(" Global best algorithm across all years (smallest ERT for each dim/function/target):\n")
print(df_best_overall.head(20))

# Optionally save to file
df_best_overall.to_csv("results/global_best_algos_bbob-biobj.csv", index=False)


 Global best algorithm across all years (smallest ERT for each dim/function/target):

    dimension  function_id  year                    best_algorithm  \
0           2            1  2016                               NaN   
1           2            1  2016      SMS-EMOA-DE_Auger_bbob-biobj   
2           2            1  2016  HMO-CMA-ES_Loshchilov_bbob-biobj   
3           2            1  2021          DMS_Brockhoff_bbob-biobj   
4           2            1  2022                     K-RVEA_Tanabe   
5           2            2  2016                               NaN   
6           2            2  2016    UP-MO-CMA-ES_Krause_bbob-biobj   
7           2            2  2016  HMO-CMA-ES_Loshchilov_bbob-biobj   
8           2            2  2022                        TPB_Tanabe   
9           2            2  2021          DMS_Brockhoff_bbob-biobj   
10          2            3  2016                               NaN   
11          2            3  2016    UP-MO-CMA-ES_Krause_bbob-biobj   
12  

### General table summing up nice stats.

- Where the best algos are located (year)
- Which algorithms are the dominant winners (counter)

This wil be used to check the results especially for the count.

In [6]:
# Count wins per year
print(df_best_overall["year"].value_counts())

# Count wins per algorithm
print(df_best_overall["best_algorithm"].value_counts().head(10))


2019    1396
2016    1155
2022     189
2021      20
Name: year, dtype: int64
HMO-CMA-ES_Loshchilov_bbob-biobj       487
GDE3-platypus_Brockhoff_bbob-biobj     188
SPEA2-platypus_Brockhoff_bbob-biobj    179
UP-MO-CMA-ES_Krause_bbob-biobj         163
RM-MEDA_Auger_bbob-biobj               111
TPB_Tanabe                              90
IBEA-platypus_Brockhoff_bbob-biobj      75
K-RVEA_Tanabe                           58
MOTPE_Tanabe                            41
COMO-316_dufosse_bbob-biobj             40
Name: best_algorithm, dtype: int64


We can see that the year where there are the most "best algos" is the year 2010. 
in addition the algo that appears the most is SLSQP+lq-CMA-ES_Hansen.

## Structure of te code below:
We are going to loop over each dimension and over each target. We will chose the target such it is the lowest and the ERT is finite. We disregard the target if the ERT is undefined.

In [7]:
import numpy as np
results = []

for dim in sorted(df_all["dimension"].unique()):
    #  Get all rows for this dimension
    df_dim = df_all[df_all["dimension"] == dim]
    
    #  Sort targets from lowest (most precise) to highest
    targets_sorted = sorted(df_dim["target"].unique())
    
    #  Find the lowest target that has at least one finite ERT
    chosen_target = None
    for t in targets_sorted:
        if np.isfinite(df_dim.loc[df_dim["target"] == t, "best_ERT"]).any():
            chosen_target = t
            break
    
    if chosen_target is None:
        # No valid ERTs for this dimension — skip
        continue

    #  Filter to that chosen target and pick the row with the smallest ERT
    df_t = df_dim[df_dim["target"] == chosen_target]
    best_row = df_t.loc[df_t["best_ERT"].idxmin(), ["dimension", "year", "best_algorithm", "target", "best_ERT"]]
    
    results.append(best_row)

#  Build summary DataFrame
df_best_by_dim = pd.DataFrame(results).reset_index(drop=True)

# Rename for clarity
df_best_by_dim.rename(columns={"target": "target_used"}, inplace=True)

print("Best algorithm per dimension (lowest usable target):")
print(df_best_by_dim)

Best algorithm per dimension (lowest usable target):
   dimension  year                       best_algorithm   target_used  \
0          2  2016       UP-MO-CMA-ES_Krause_bbob-biobj  1.000000e-08   
1          3  2016       UP-MO-CMA-ES_Krause_bbob-biobj  1.000000e-08   
2          5  2019          COMO-316_dufosse_bbob-biobj  1.000000e-08   
3         10  2016       UP-MO-CMA-ES_Krause_bbob-biobj  1.000000e-08   
4         20  2019          COMO-316_dufosse_bbob-biobj  1.000000e-08   
5         40  2019  SPEA2-platypus_Brockhoff_bbob-biobj  1.000000e-08   

       best_ERT  
0  8.159804e+05  
1  1.716453e+06  
2  2.350528e+06  
3  8.553205e+06  
4  1.894075e+07  
5  3.774390e+05  


# To Do for November 6

best 10 per dimension: see photo 
eaach dimension the most contributing: like summing up for the different dimensions
different bbob

you give the number of algo that you want, like if you wnat 10 algos, you may to go down several lines in the table to get all the data. maybe in the next line of the table you have more than one more and so you have to decide, so in fact you don't just give plus one algo, you give back to exact number that are in the following table/ or you decide a certain way on how to chose that extra algo. / or you look at the one that has the highest sum: for choosing the next best algo.

## The goal is to nicely present our results in a table summing up which are the best algorithms over all. 

Here we are building a table that sums up the 10 best algorithms for each dimension: there are 6 different dimensions. We can also see the count for how many times that algorithm was the best in that specific dimension ( remember that we are looping over all the different 24 functions)


# how are the best algos found ? are we looping over dimension, function ad target ? and ow is the target determined ?

### Structure of the code:

1) Count how many times each algo appears as best within each dimension, remember that this also loops over all the different functions. 
2) Sort this count
3) For each dimension, keep rank of algorithms by count

In [15]:
# === 4. Aggregate: count how many times each algorithm was best per dimension ===

# Count how many times each algorithm appears as best within each dimension
algo_counts = (
    df_best_overall
    .groupby(["dimension", "best_algorithm"])
    .size()  # count occurrences
    .reset_index(name="count")
)

# Sort within each dimension by count descending
algo_counts = (
    algo_counts
    .sort_values(["dimension", "count"], ascending=[True, False])
)

# For convenience: for each dimension, keep rank of algorithms by count
algo_counts["rank"] = algo_counts.groupby("dimension")["count"].rank(method="first", ascending=False)

# === 5. (Optional) Pivot into a "table" format for top 10 ===

# For each dimension, show the 10 top algorithms with their counts
top10_by_dim = (
    algo_counts
    .groupby("dimension")
    .head(10)  # take 10 first per dimension
)

# Display as a clean summary
print("\n Top 10 algorithms per dimension by number of 'best' occurrences:\n")
for dim in sorted(top10_by_dim["dimension"].unique()):
    print(f"=== Dimension {dim} ===")
    subset = top10_by_dim[top10_by_dim["dimension"] == dim][["best_algorithm", "count"]]
    print(subset.to_string(index=False))
    print()



 Top 10 algorithms per dimension by number of 'best' occurrences:

=== Dimension 2 ===
                     best_algorithm  count
SPEA2-platypus_Brockhoff_bbob-biobj     53
     UP-MO-CMA-ES_Krause_bbob-biobj     51
   HMO-CMA-ES_Loshchilov_bbob-biobj     50
                      K-RVEA_Tanabe     26
 IBEA-platypus_Brockhoff_bbob-biobj     16
                       MOTPE_Tanabe     15
  MO-DIRECT-HV-Rank_Wong_bbob-biobj     14
           RM-MEDA_Auger_bbob-biobj     14
                         TPB_Tanabe     13
           DMS_Brockhoff_bbob-biobj     12

=== Dimension 3 ===
                     best_algorithm  count
   HMO-CMA-ES_Loshchilov_bbob-biobj     82
SPEA2-platypus_Brockhoff_bbob-biobj     38
     UP-MO-CMA-ES_Krause_bbob-biobj     33
                      K-RVEA_Tanabe     20
                         TPB_Tanabe     18
 IBEA-platypus_Brockhoff_bbob-biobj     16
 GDE3-platypus_Brockhoff_bbob-biobj     15
           RM-MEDA_Auger_bbob-biobj     14
                       MOTPE_Ta

This is an additional interesting table ( mainly to verify that the above results are correct). We only take the 10 best algorithms that have the highest occurence of "best" performances. Then for each algorithm we repertoriate for each dimensin the number times it is the best. Their aggregates match with the above resuts. 

In [9]:
pivot_table = algo_counts.pivot(index="best_algorithm", columns="dimension", values="count").fillna(0).astype(int)
pivot_table["Total"] = pivot_table.sum(axis=1)
pivot_table = pivot_table.sort_values("Total", ascending=False)

print("\n Overall frequency of being best by algorithm and dimension (top 10):\n")
print(pivot_table.head(10))



 Overall frequency of being best by algorithm and dimension (top 10):

dimension                             2   3   5   10   20   40  Total
best_algorithm                                                       
HMO-CMA-ES_Loshchilov_bbob-biobj     50  82  99  125  131    0    487
GDE3-platypus_Brockhoff_bbob-biobj   11  15  17   17   19  109    188
SPEA2-platypus_Brockhoff_bbob-biobj  53  38  30   18   14   26    179
UP-MO-CMA-ES_Krause_bbob-biobj       51  33  38   20   21    0    163
RM-MEDA_Auger_bbob-biobj             14  14   3    2    1   77    111
TPB_Tanabe                           13  18  19   19   21    0     90
IBEA-platypus_Brockhoff_bbob-biobj   16  16  14   18   11    0     75
K-RVEA_Tanabe                        26  20  10    1    1    0     58
MOTPE_Tanabe                         15  13  10    3    0    0     41
COMO-316_dufosse_bbob-biobj           1   4  13   15    7    0     40


Now we are interested in presenting the data in a readabe table. The columns of this table are the different dimmension, and the lines of the table represent the ranking of the best algorithms. In the first line we will have the best algorithm for each dimension. The second line will represnt the 2nd best algorithms for each dimension etc... 

# Attention here we are aggreggating over all the different targets.

## Question: here what are the target precisions ? are they necessary the lowest ? or the lowest st the ERT is still defined ? 

### Explanation of the code

1) again we are counting how many times it was the best over each dimension. best = the one that has teh apperances, this will be the rank of an algo. 
2) set the table up: rank = rows and the dimension are on the columns


In [10]:
# === 4. Aggregate: count how many times each algorithm was best per dimension ===
algo_counts = (
    df_best_overall
    .groupby(["dimension", "best_algorithm"])
    .size()
    .reset_index(name="count")
)

# Sort within each dimension by how often each algo was best
algo_counts = algo_counts.sort_values(["dimension", "count"], ascending=[True, False])

# Add ranking per dimension
algo_counts["rank"] = algo_counts.groupby("dimension")["count"].rank(method="first", ascending=False).astype(int)

# === 5. Pivot: rows = rank (1st best, 2nd best...), columns = dimension, values = algorithm name ===
algo_ranking_table = algo_counts.pivot(index="rank", columns="dimension", values="best_algorithm")

# Sort columns (dimensions) in ascending order
algo_ranking_table = algo_ranking_table.reindex(sorted(algo_ranking_table.columns), axis=1)

# === 6. Display neatly ===
print("\n Ranking of best algorithms per dimension:\n")
from IPython.display import display
display(algo_ranking_table.head(10))  # show top 10 ranks



 Ranking of best algorithms per dimension:



dimension,2,3,5,10,20,40
rank,,,,,,
1,SPEA2-platypus_Brockhoff_bbob-biobj,HMO-CMA-ES_Loshchilov_bbob-biobj,HMO-CMA-ES_Loshchilov_bbob-biobj,HMO-CMA-ES_Loshchilov_bbob-biobj,HMO-CMA-ES_Loshchilov_bbob-biobj,GDE3-platypus_Brockhoff_bbob-biobj
2,UP-MO-CMA-ES_Krause_bbob-biobj,SPEA2-platypus_Brockhoff_bbob-biobj,UP-MO-CMA-ES_Krause_bbob-biobj,UP-MO-CMA-ES_Krause_bbob-biobj,TPB_Tanabe,RM-MEDA_Auger_bbob-biobj
3,HMO-CMA-ES_Loshchilov_bbob-biobj,UP-MO-CMA-ES_Krause_bbob-biobj,SPEA2-platypus_Brockhoff_bbob-biobj,TPB_Tanabe,UP-MO-CMA-ES_Krause_bbob-biobj,SPEA2-platypus_Brockhoff_bbob-biobj
4,K-RVEA_Tanabe,K-RVEA_Tanabe,TPB_Tanabe,IBEA-platypus_Brockhoff_bbob-biobj,GDE3-platypus_Brockhoff_bbob-biobj,MOEAD-platypus_Brockhoff_bbob-biobj
5,IBEA-platypus_Brockhoff_bbob-biobj,TPB_Tanabe,GDE3-platypus_Brockhoff_bbob-biobj,SPEA2-platypus_Brockhoff_bbob-biobj,SPEA2-platypus_Brockhoff_bbob-biobj,SMS-EMOA-DE_Auger_bbob-biobj
6,MOTPE_Tanabe,IBEA-platypus_Brockhoff_bbob-biobj,IBEA-platypus_Brockhoff_bbob-biobj,GDE3-platypus_Brockhoff_bbob-biobj,IBEA-platypus_Brockhoff_bbob-biobj,NaN
7,MO-DIRECT-HV-Rank_Wong_bbob-biobj,GDE3-platypus_Brockhoff_bbob-biobj,COMO-316_dufosse_bbob-biobj,COMO-316_dufosse_bbob-biobj,COMO-10_dufosse_bbob-biobj,NaN
8,RM-MEDA_Auger_bbob-biobj,RM-MEDA_Auger_bbob-biobj,COMO-100_dufosse_bbob-biobj,COMO-100_dufosse_bbob-biobj,COMO-316_dufosse_bbob-biobj,NaN
9,TPB_Tanabe,MOTPE_Tanabe,K-RVEA_Tanabe,COMO-3_dufosse_bbob-biobj,COMO-100_dufosse_bbob-biobj,NaN


In [11]:
def get_top_algorithms(algo_ranking_table, N_min=6):
    """
    Collect algorithms from top ranks (rows) across all dimensions
    until at least N_min unique algorithms are found.
    Returns the set of algorithms and the actual count.
    """
    seen_algos = set()
    row_idx = 0
    
    # Keep adding rows until we reach at least N_min unique algos
    while len(seen_algos) < N_min and row_idx < len(algo_ranking_table):
        row_algos = algo_ranking_table.iloc[row_idx].dropna().unique()
        seen_algos.update(row_algos)
        row_idx += 1

    # Result
    algo_list = sorted(seen_algos)
    print(f"➡️ Requested {N_min} best algorithms.")
    print(f"✅ Returning {len(algo_list)} unique algorithms:")
    return algo_list

# Example usage:
best6 = get_top_algorithms(algo_ranking_table, N_min=6)
print(best6)

best7 = get_top_algorithms(algo_ranking_table, N_min=7)
print(best7)


➡️ Requested 6 best algorithms.
✅ Returning 6 unique algorithms:
['GDE3-platypus_Brockhoff_bbob-biobj', 'HMO-CMA-ES_Loshchilov_bbob-biobj', 'RM-MEDA_Auger_bbob-biobj', 'SPEA2-platypus_Brockhoff_bbob-biobj', 'TPB_Tanabe', 'UP-MO-CMA-ES_Krause_bbob-biobj']
➡️ Requested 7 best algorithms.
✅ Returning 9 unique algorithms:
['GDE3-platypus_Brockhoff_bbob-biobj', 'HMO-CMA-ES_Loshchilov_bbob-biobj', 'IBEA-platypus_Brockhoff_bbob-biobj', 'K-RVEA_Tanabe', 'MOEAD-platypus_Brockhoff_bbob-biobj', 'RM-MEDA_Auger_bbob-biobj', 'SPEA2-platypus_Brockhoff_bbob-biobj', 'TPB_Tanabe', 'UP-MO-CMA-ES_Krause_bbob-biobj']


We have noticed that the best algorithms come as batches of 6 algorithms per line. So hence when we are asking for the N best algorithms, where N<6, it is complicated to choose which algorithms to return. This is why we will set a minimum of 6 best algorithms, one for each dimension. Now we may be asked for more than 6 best algrithms. 


For example let's say we are asking for the 8 best algorithms. In this case we will go through the following lines of the table. There are several cases we need to consider to choose these 8 best algorithms.

- One scenario, is that in the following line, there are the names of algorithms that were already mentioned in the first line. In this case we don't add their name again. 
- The second scenario is we are in fact not able to return the names of 8 algorithms, because all the algorithms have performed equally for their respective dimension. Instead of choosing only 8, we will return the names of all the algorithms that are mentioned and are on the same line. We also notify that we are not returning 8 algorithms, but a bit more, here 10.
- A further step would be to really restrict the number to 8. This means we need to find a way to classify the algorithms that are on the same line. An easy way is to select the one that has the highest occurence of "best" algorithm. 

In [14]:
def get_top_algorithms(algo_ranking_table, N_min):
    """
    Collect algorithms from top ranks (rows) across all dimensions
    until at least N_min unique algorithms are found.
    """
    seen_algos = set()
    row_idx = 0

    # Keep adding rows until we have at least N_min unique algorithms
    while len(seen_algos) < N_min and row_idx < len(algo_ranking_table):
        row_algos = algo_ranking_table.iloc[row_idx].dropna().unique()
        seen_algos.update(row_algos)
        row_idx += 1

    algo_list = sorted(seen_algos)

    print(f"\n Requested {N_min} best algorithms.")
    print(f" Returning {len(algo_list)} unique algorithms (reached rank {row_idx}).\n")
    print(" List of selected algorithms:\n")
    for algo in algo_list:
        print(f" - {algo}")

    return algo_list


# === Interactive part ===
try:
    # Ask user for number of desired algorithms
    N_input = int(input("How many best algorithms do you want? "))
    #if N_input < 6:
        #print(" Minimum number of best algorithms is 6. Using N=6.")
       # N_input = 6

    # Compute and display
    best_algos = get_top_algorithms(algo_ranking_table, N_min=N_input)

except ValueError:
    print("Invalid input. Please enter an integer number (e.g., 6 or 7).")


How many best algorithms do you want? 2

 Requested 2 best algorithms.
 Returning 3 unique algorithms (reached rank 1).

 List of selected algorithms:

 - GDE3-platypus_Brockhoff_bbob-biobj
 - HMO-CMA-ES_Loshchilov_bbob-biobj
 - SPEA2-platypus_Brockhoff_bbob-biobj


In [ ]:
def make_dimension_ranking_table(df_best_overall, target):
    """
    Build the dimension-wise ranking table:
    Rows = rank (1st best, 2nd best...)
    Columns = dimension (2, 3, 5, 10, 20, 40, ...)
    Values = algorithm name

    Counts how many times each algorithm was best for (fct_id, target)
    and ranks algorithms separately inside each dimension.
    """
    # Filter for chosen target
    df_t = df_best_overall[df_best_overall["target"] == target].copy()
    
    if df_t.empty:
        raise ValueError(f"No data found for target precision = {target}")

    # === 4. Aggregate: how many times each algorithm was best per dimension ===
    algo_counts = (
        df_t.groupby(["dimension", "best_algorithm"])
        .size()
        .reset_index(name="count")
    )

    algo_counts = algo_counts.sort_values(
        ["dimension", "count"],
        ascending=[True, False]
    )

    # Add ranking inside each dimension
    algo_counts["rank"] = (
        algo_counts.groupby("dimension")["count"]
        .rank(method="first", ascending=False)
        .astype(int)
    )

    # === 5. Pivot: rank × dimension ===
    ranking_table = algo_counts.pivot(
        index="rank",
        columns="dimension",
        values="best_algorithm"
    )

    ranking_table = ranking_table.reindex(
        sorted(ranking_table.columns),
        axis=1
    )

    print(f"\n📌 Ranking table for target precision = {target}\n")
    from IPython.display import display
    display(ranking_table)

    return ranking_table
try:
    print("\nAvailable targets:")
    print(df_best_overall["target"].unique())

    target_choice = float(input("\nChoose a target precision (e.g., 1e-8): "))

    algo_ranking_table = make_dimension_ranking_table(df_best_overall, target_choice)

except ValueError:
    print("Invalid input. Please enter a numeric target (e.g., 1e-8).")


# Plotting the results.

In [10]:
import cocopp

cocopp.main(['GDE3-platypus_Brockhoff_bbob-biobj','HMO-CMA-ES_Loshchilov_bbob-biobj','IBEA-platypus_Brockhoff_bbob-biobj','K-RVEA_Tanabe','MOEAD-platypus_Brockhoff_bbob-biobj','RM-MEDA_Auger_bbob-biobj','SPEA2-platypus_Brockhoff_bbob-biobj','TPB_Tanabe','UP-MO-CMA-ES_Krause_bbob-biobj'])

Post-processing (2+)
  Using 9 data sets:
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-biobj\2019\GDE3-platypus_Brockhoff_bbob-biobj.tgz
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-biobj\2016\HMO-CMA-ES_Loshchilov_bbob-biobj.tgz
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-biobj\2019\IBEA-platypus_Brockhoff_bbob-biobj.tgz
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-biobj\2022\K-RVEA_Tanabe.tgz
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-biobj\2019\MOEAD-platypus_Brockhoff_bbob-biobj.tgz
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-biobj\2016\RM-MEDA_Auger_bbob-biobj.tgz
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-biobj\2019\SPEA2-platypus_Brockhoff_bbob-biobj.tgz
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-biobj\2022\TPB_Tanabe.tgz
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-biobj\2016\UP-MO-CMA-ES_Krau

C:\Users\elsaf\anaconda3\Lib\site-packages\cocopp\pproc.py:3895: UserWarning:  Reference values for the algorithm 'C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-biobj\2016\HMO-CMA-ES_Loshchilov_bbob-biobj.tgz' are different!
  warnings.warn(" Reference values for the algorithm '%s' are different!" % alg)


  Data consistent according to consistency_check() in pproc.DataSet
  using: C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-biobj\2019\IBEA-platypus_Brockhoff_bbob-biobj.tgz


C:\Users\elsaf\anaconda3\Lib\site-packages\cocopp\pproc.py:3895: UserWarning:  Reference values for the algorithm 'C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-biobj\2019\IBEA-platypus_Brockhoff_bbob-biobj.tgz' are different!
  warnings.warn(" Reference values for the algorithm '%s' are different!" % alg)
C:\Users\elsaf\anaconda3\Lib\site-packages\cocopp\pproc.py:3895: UserWarning:  Reference values for the algorithm 'C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-biobj\2022\K-RVEA_Tanabe.tgz' are different!
  warnings.warn(" Reference values for the algorithm '%s' are different!" % alg)


  using: C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-biobj\2022\K-RVEA_Tanabe.tgz
  Data consistent according to consistency_check() in pproc.DataSet
  using: C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-biobj\2019\MOEAD-platypus_Brockhoff_bbob-biobj.tgz
  Data consistent according to consistency_check() in pproc.DataSet
  using: C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-biobj\2016\RM-MEDA_Auger_bbob-biobj.tgz
  Data consistent according to consistency_check() in pproc.DataSet
  using: C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-biobj\2019\SPEA2-platypus_Brockhoff_bbob-biobj.tgz


...

C:\Users\elsaf\anaconda3\Lib\site-packages\cocopp\testbedsettings.py:126: UserWarning:  Reference values for the algorithm 'HMO-CMA-ES_Loshchilov_bbob-biobj' are different from the algorithm 'GDE3-platypus_Brockhoff_bbob-biobj'
  warnings.warn(" Reference values for the algorithm '%s' are different from the algorithm '%s'"
C:\Users\elsaf\anaconda3\Lib\site-packages\cocopp\testbedsettings.py:126: UserWarning:  Reference values for the algorithm 'K-RVEA_Tanabe' are different from the algorithm 'GDE3-platypus_Brockhoff_bbob-biobj'
  warnings.warn(" Reference values for the algorithm '%s' are different from the algorithm '%s'"
C:\Users\elsaf\anaconda3\Lib\site-packages\cocopp\testbedsettings.py:126: UserWarning:  Reference values for the algorithm 'MOTPE_Tanabe' are different from the algorithm 'GDE3-platypus_Brockhoff_bbob-biobj'
  warnings.warn(" Reference values for the algorithm '%s' are different from the algorithm '%s'"
C:\Users\elsaf\anaconda3\Lib\site-packages\cocopp\testbedsetting

  Will generate output data in folder ppdata\biobj-ext_GDE3-_HMO-C_IBEA-_K-RVE_MOEAD_RM-ME_SPEA2_et_al_111921h0216
    this might take several minutes.
ECDF graphs per noise group...
Loading best algorithm data from refalgs/best2016-bbob-biobj.tar.gz ...
    archive extracted to folder C:\Users\elsaf\anaconda3\Lib\site-packages\cocopp\refalgs\.extracted_best2016-bbob-biobj ...
  Data consistent according to consistency_check() in pproc.DataSet
  using: C:\Users\elsaf\anaconda3\Lib\site-packages\cocopp\refalgs/best2016-bbob-biobj.tar.gz


C:\Users\elsaf\anaconda3\Lib\site-packages\cocopp\testbedsettings.py:126: UserWarning:  Reference values for the algorithm 'refalgs/best2016-bbob-biobj' are different from the algorithm 'GDE3-platypus_Brockhoff_bbob-biobj'
  warnings.warn(" Reference values for the algorithm '%s' are different from the algorithm '%s'"


  done (Wed Nov 19 21:02:22 2025).
  done (Wed Nov 19 21:02:43 2025).
ECDF graphs per function group...
  done (Wed Nov 19 21:04:44 2025).
ECDF graphs per function...
  done (Wed Nov 19 21:10:31 2025).
Generating comparison tables...
[DataSet(K-RVEA_Tanabe on f1 2-D), DataSet(MOTPE_Tanabe on f1 2-D), DataSet(TPB_Tanabe on f1 2-D)]


Exception: There is more than a single entry associated with folder C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-biobj\2022\K-RVEA_Tanabe.tgz on 2-D f1.

In [11]:
cocopp.main(['GDE3-platypus_Brockhoff_bbob-biobj'])
             
             #'GDE3-platypus_Brockhoff_bbob-biobj','HMO-CMA-ES_Loshchilov_bbob-biobj','IBEA-platypus_Brockhoff_bbob-biobj','K-RVEA_Tanabe','MOEAD-platypus_Brockhoff_bbob-biobj','RM-MEDA_Auger_bbob-biobj','SPEA2-platypus_Brockhoff_bbob-biobj','TPB_Tanabe','UP-MO-CMA-ES_Krause_bbob-biobj'])

Post-processing (1)
  Using 1 data set:
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-biobj\2019\GDE3-platypus_Brockhoff_bbob-biobj.tgz

Post-processing (1)
  loading data...
  Data consistent according to consistency_check() in pproc.DataSet
  Will generate output data in folder ppdata\GDE3-platypus_Brockhoff_bbob-biobj_111922h1539
    this might take several minutes.
Scaling figures...
  done (Wed Nov 19 22:21:41 2025).
Generating LaTeX tables...
  done (Wed Nov 19 22:21:43 2025).
ECDF graphs...
  done (Wed Nov 19 22:22:13 2025).
ECDF graphs per function...
  done (Wed Nov 19 22:25:33 2025).
ERT loss ratio figures and tables...
  done (Wed Nov 19 22:25:33 2025).
Output data written to folder C:\Users\elsaf\PRL project on jupyter\ppdata\GDE3-platypus_Brockhoff_bbob-biobj_111922h1539
Setting changes in `cocopp.genericsettings` compared to default:
    simulated_runlength_bootstrap_sample_size: from 30 to 10.098990100989901
    foreground_algorithm_list: from [] to ['

DictAlg([(('GDE3-platypus_Brockhoff_bbob-biobj', ''),
          [DataSet(GDE3-platypus_Brockhoff_bbob-biobj on f1 2-D),
           DataSet(GDE3-platypus_Brockhoff_bbob-biobj on f2 2-D),
           DataSet(GDE3-platypus_Brockhoff_bbob-biobj on f3 2-D),
           DataSet(GDE3-platypus_Brockhoff_bbob-biobj on f4 2-D),
           DataSet(GDE3-platypus_Brockhoff_bbob-biobj on f5 2-D),
           DataSet(GDE3-platypus_Brockhoff_bbob-biobj on f6 2-D),
           DataSet(GDE3-platypus_Brockhoff_bbob-biobj on f7 2-D),
           DataSet(GDE3-platypus_Brockhoff_bbob-biobj on f8 2-D),
           DataSet(GDE3-platypus_Brockhoff_bbob-biobj on f9 2-D),
           DataSet(GDE3-platypus_Brockhoff_bbob-biobj on f10 2-D),
           DataSet(GDE3-platypus_Brockhoff_bbob-biobj on f11 2-D),
           DataSet(GDE3-platypus_Brockhoff_bbob-biobj on f12 2-D),
           DataSet(GDE3-platypus_Brockhoff_bbob-biobj on f13 2-D),
           DataSet(GDE3-platypus_Brockhoff_bbob-biobj on f14 2-D),
           DataSe

In [12]:
cocopp.main(['HMO-CMA-ES_Loshchilov_bbob-biobj'])

Post-processing (1)
  Using 1 data set:
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-biobj\2016\HMO-CMA-ES_Loshchilov_bbob-biobj.tgz

Post-processing (1)
  loading data...
  Data consistent according to consistency_check() in pproc.DataSet
  Will generate output data in folder ppdata\HMO-CMA-ES_Loshchilov_bbob-biobj_111922h2533
    this might take several minutes.
Scaling figures...
Loading best algorithm data from refalgs/best2016-bbob-biobj.tar.gz ...
  using: C:\Users\elsaf\anaconda3\Lib\site-packages\cocopp\refalgs/best2016-bbob-biobj.tar.gz
  done (Wed Nov 19 22:25:54 2025).
  done (Wed Nov 19 22:26:47 2025).
Generating LaTeX tables...
  done (Wed Nov 19 22:26:49 2025).
ECDF graphs...
  done (Wed Nov 19 22:27:03 2025).
ECDF graphs per function...
  done (Wed Nov 19 22:29:20 2025).
ERT loss ratio figures and tables...


C:\Users\elsaf\anaconda3\Lib\site-packages\cocopp\pplogloss.py:776: RuntimeWarning: divide by zero encountered in log10
  ydata.append(np.log10(list(data[f][i] for f in data)))
C:\Users\elsaf\anaconda3\Lib\site-packages\cocopp\pplogloss.py:776: RuntimeWarning: divide by zero encountered in log10
  ydata.append(np.log10(list(data[f][i] for f in data)))
C:\Users\elsaf\anaconda3\Lib\site-packages\cocopp\pplogloss.py:776: RuntimeWarning: divide by zero encountered in log10
  ydata.append(np.log10(list(data[f][i] for f in data)))
C:\Users\elsaf\anaconda3\Lib\site-packages\cocopp\pplogloss.py:776: RuntimeWarning: divide by zero encountered in log10
  ydata.append(np.log10(list(data[f][i] for f in data)))
C:\Users\elsaf\anaconda3\Lib\site-packages\cocopp\pplogloss.py:776: RuntimeWarning: divide by zero encountered in log10
  ydata.append(np.log10(list(data[f][i] for f in data)))


  done (Wed Nov 19 22:29:34 2025).
Output data written to folder C:\Users\elsaf\PRL project on jupyter\ppdata\HMO-CMA-ES_Loshchilov_bbob-biobj_111922h2533
Setting changes in `cocopp.genericsettings` compared to default:
    simulated_runlength_bootstrap_sample_size: from 30 to 10.098990100989901
    foreground_algorithm_list: from [] to ['C:\\Users\\elsaf\\Ap...
ALL done (Wed Nov 19 22:29:35 2025).


DictAlg([(('HMO-CMA-ES_Loshchilov_bbob-biobj', ''),
          [DataSet(HMO-CMA-ES_Loshchilov_bbob-biobj on f1 2-D),
           DataSet(HMO-CMA-ES_Loshchilov_bbob-biobj on f2 2-D),
           DataSet(HMO-CMA-ES_Loshchilov_bbob-biobj on f3 2-D),
           DataSet(HMO-CMA-ES_Loshchilov_bbob-biobj on f4 2-D),
           DataSet(HMO-CMA-ES_Loshchilov_bbob-biobj on f5 2-D),
           DataSet(HMO-CMA-ES_Loshchilov_bbob-biobj on f6 2-D),
           DataSet(HMO-CMA-ES_Loshchilov_bbob-biobj on f7 2-D),
           DataSet(HMO-CMA-ES_Loshchilov_bbob-biobj on f8 2-D),
           DataSet(HMO-CMA-ES_Loshchilov_bbob-biobj on f9 2-D),
           DataSet(HMO-CMA-ES_Loshchilov_bbob-biobj on f10 2-D),
           DataSet(HMO-CMA-ES_Loshchilov_bbob-biobj on f11 2-D),
           DataSet(HMO-CMA-ES_Loshchilov_bbob-biobj on f12 2-D),
           DataSet(HMO-CMA-ES_Loshchilov_bbob-biobj on f13 2-D),
           DataSet(HMO-CMA-ES_Loshchilov_bbob-biobj on f14 2-D),
           DataSet(HMO-CMA-ES_Loshchilov_bbob-b

In [13]:
cocopp.main(['IBEA-platypus_Brockhoff_bbob-biobj'])

Post-processing (1)
  Using 1 data set:
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-biobj\2019\IBEA-platypus_Brockhoff_bbob-biobj.tgz

Post-processing (1)
  loading data...
  Data consistent according to consistency_check() in pproc.DataSet
  Will generate output data in folder ppdata\IBEA-platypus_Brockhoff_bbob-biobj_111922h2935
    this might take several minutes.
Scaling figures...
  done (Wed Nov 19 22:34:43 2025).
Generating LaTeX tables...
  done (Wed Nov 19 22:34:45 2025).
ECDF graphs...
  done (Wed Nov 19 22:35:07 2025).
ECDF graphs per function...
  done (Wed Nov 19 22:37:50 2025).
ERT loss ratio figures and tables...
  done (Wed Nov 19 22:37:50 2025).
Output data written to folder C:\Users\elsaf\PRL project on jupyter\ppdata\IBEA-platypus_Brockhoff_bbob-biobj_111922h2935
Setting changes in `cocopp.genericsettings` compared to default:
    simulated_runlength_bootstrap_sample_size: from 30 to 10.098990100989901
    foreground_algorithm_list: from [] to ['

DictAlg([(('IBEA-platypus_Brockhoff_bbob-biobj', ''),
          [DataSet(IBEA-platypus_Brockhoff_bbob-biobj on f1 2-D),
           DataSet(IBEA-platypus_Brockhoff_bbob-biobj on f2 2-D),
           DataSet(IBEA-platypus_Brockhoff_bbob-biobj on f3 2-D),
           DataSet(IBEA-platypus_Brockhoff_bbob-biobj on f4 2-D),
           DataSet(IBEA-platypus_Brockhoff_bbob-biobj on f5 2-D),
           DataSet(IBEA-platypus_Brockhoff_bbob-biobj on f6 2-D),
           DataSet(IBEA-platypus_Brockhoff_bbob-biobj on f7 2-D),
           DataSet(IBEA-platypus_Brockhoff_bbob-biobj on f8 2-D),
           DataSet(IBEA-platypus_Brockhoff_bbob-biobj on f9 2-D),
           DataSet(IBEA-platypus_Brockhoff_bbob-biobj on f10 2-D),
           DataSet(IBEA-platypus_Brockhoff_bbob-biobj on f11 2-D),
           DataSet(IBEA-platypus_Brockhoff_bbob-biobj on f12 2-D),
           DataSet(IBEA-platypus_Brockhoff_bbob-biobj on f13 2-D),
           DataSet(IBEA-platypus_Brockhoff_bbob-biobj on f14 2-D),
           DataSe